In [1]:

import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import numpy as np
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
# from tenacity import retry, stop_after_attempt, wait_exponential
import re
import gc
import matplotlib.pyplot as plt
from tqdm import tqdm
api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

# 한글 폰트 설정
import matplotlib.font_manager as fm
import matplotlib as mpl

# 한글 폰트 경로 설정 (맥OS 기준)
font_path = '/System/Library/Fonts/AppleSDGothicNeo.ttc'  # 맥OS의 기본 한글 폰트
font_prop = fm.FontProperties(fname=font_path)

# matplotlib 기본 폰트 설정
plt.rc('font', family=font_prop.get_name())
mpl.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 폰트 확인
print(f"설정된 폰트: {font_prop.get_name()}")
print(f"사용 가능한 한글 폰트:")
for font in fm.findSystemFonts():
    if 'gothic' in font.lower() or 'gulim' in font.lower() or 'malgun' in font.lower() or 'batang' in font.lower():
        print(f" - {font}")

# pd.set_option('display.max_rows', 100)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', 100)
# pd.set_option('display.max_colwidth', None)


df = pd.read_hdf('/Users/nam-yeong/git/prj_centum/gpt_word/final_result/preprocessed_final_df.h5', 
                 key='df')

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_75611/2619118992.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


설정된 폰트: Apple SD Gothic Neo
사용 가능한 한글 폰트:
 - /System/Library/Fonts/Supplemental/AppleGothic.ttf
 - /System/Library/Fonts/AppleSDGothicNeo.ttc
 - /System/Library/Fonts/Supplemental/NotoSansGothic-Regular.ttf


## 피처 생성
y값 기준 정의
- vas 처음 대비 20% 미만으로 감소한 환자, 
- cmo 43mm 이상으로 증가한 환자, 
- mmo-cmo 차이가 0이 된 환자 등

- after 의 경우 Null 값이 너무 많음. drop (내판단)

### 완치 기준 설정 및 평가 시스템

In [2]:
# 사용
def calculate_composite_recovery_score(patient_data):
    """
    여러 임상 지표를 통합한 복합 완치 점수 계산
    
    Parameters:
    -----------
    patient_data : dict or pandas.Series
        환자의 첫 방문과 마지막 방문 데이터가 포함된 객체
    
    Returns:
    --------
    float
        0-100 사이의 복합 완치 점수
    dict
        각 기준별 점수와 가중치가 포함된 세부 정보
    """
    # 임상적 중요도에 따른 가중치 설정
    weights = {
        'vas_improvement': 0.35,  # 통증 감소가 가장 중요
        'cmo_improvement': 0.30,  # 편안한 개구량 개선
        'mmo_cmo_gap': 0.20,      # MMO-CMO 차이
        'additional_factors': 0.15  # 기타 요소(소리, 압흔 등)
    }
    
    scores = {}
    
    # 1. VAS 개선 점수 (통증 감소) - 0-100점
    if 'CC_vas_성장률' in patient_data and pd.notna(patient_data['CC_vas_성장률']):
        vas_change = patient_data['CC_vas_성장률']
        # 통증이 완전히 사라짐(100% 감소)이 최고 점수, 50% 감소는 중간 점수
        if vas_change <= -100:  # 통증 완전 소실
            scores['vas_improvement'] = 100
        elif vas_change >= 0:   # 통증 증가 또는 변화 없음
            scores['vas_improvement'] = 0
        else:  # 통증 감소 (선형 스케일링)
            scores['vas_improvement'] = min(100, abs(vas_change) * 100 / 100)
    else:
        scores['vas_improvement'] = 0
    
    # 2. CMO 개선 점수 (편안한 개구량) - 0-100점
    if '마지막방문_CMO_before' in patient_data and pd.notna(patient_data['마지막방문_CMO_before']):
        cmo_final = patient_data['마지막방문_CMO_before']
        cmo_initial = patient_data.get('첫방문_CMO_before', 0)
        
        # 43mm 이상은 완전 회복으로 간주
        if cmo_final >= 43:
            scores['cmo_improvement'] = 100
        elif cmo_final <= cmo_initial:  # 개선 없음
            scores['cmo_improvement'] = 0
        else:  # 개선은 있으나 43mm 미만 (선형 스케일링)
            # 정상 범위(43mm)까지의 진전도 계산
            progress_to_normal = (cmo_final - cmo_initial) / (43 - cmo_initial) * 100
            scores['cmo_improvement'] = min(100, max(0, progress_to_normal))
    else:
        scores['cmo_improvement'] = 0
    
    # 3. MMO-CMO 차이 점수 - 0-100점
    if 'dif_MMO_CMO_성장률' in patient_data and pd.notna(patient_data['dif_MMO_CMO_성장률']):
        mmo_cmo_change = patient_data['dif_MMO_CMO_성장률']
        mmo_cmo_final = patient_data.get('마지막방문_dif_MMO_CMO', float('inf'))
        
        # MMO-CMO=0이 이상적이나, 일반적으로 작은 양수값이 정상
        if mmo_cmo_final == 0:
            scores['mmo_cmo_gap'] = 100
        elif mmo_cmo_change <= -50:  # 50% 이상 감소는 좋은 징후
            scores['mmo_cmo_gap'] = 75
        elif mmo_cmo_change < 0:  # 감소 경향은 긍정적
            scores['mmo_cmo_gap'] = 50 + abs(mmo_cmo_change) / 2
        else:  # 증가 추세는 부정적
            scores['mmo_cmo_gap'] = max(0, 50 - mmo_cmo_change/2)
    else:
        scores['mmo_cmo_gap'] = 0
    
    # 4. 추가 요소들 통합 (소리, 혀/점막 압흔, 편위 등)
    additional_scores = []
    
    # 턱 소리 감소
    if 'Noise_Intensity' in patient_data and pd.notna(patient_data['Noise_Intensity']):
        noise_intensity = patient_data['Noise_Intensity']
        if noise_intensity == 0:
            additional_scores.append(100)  # 소리 없음
        elif noise_intensity <= 1:
            additional_scores.append(75)   # 경미한 소리
        elif noise_intensity <= 2:
            additional_scores.append(50)   # 중간 정도 소리
        else:
            additional_scores.append(25)   # 심한 소리
    
    # 혀/점막 압흔 감소
    for ridging in ['Tongue_ridging_Intensity', 'Mucosal_ridging_Intensity']:
        if ridging in patient_data and pd.notna(patient_data[ridging]):
            intensity = patient_data[ridging]
            if intensity == 0:
                additional_scores.append(100)
            elif intensity <= 1:
                additional_scores.append(75)
            elif intensity <= 2:
                additional_scores.append(50)
            else:
                additional_scores.append(25)
    
    # 추가 요소 평균 점수
    if additional_scores:
        scores['additional_factors'] = sum(additional_scores) / len(additional_scores)
    else:
        scores['additional_factors'] = 50  # 데이터 없는 경우 중간값 할당
    
    # 최종 복합 점수 계산 (가중치 적용)
    composite_score = sum(score * weights[key] for key, score in scores.items())
    
    # 점수에 따른 회복 등급 할당
    recovery_grade = assign_recovery_grade(composite_score)
    
    return composite_score, {'scores': scores, 'weights': weights, 'recovery_grade': recovery_grade}

def assign_recovery_grade(score):
    """복합 완치 점수에 따른 회복 등급 할당"""
    if score >= 90:
        return "완전 회복 (Complete Recovery)"
    elif score >= 75:
        return "상당한 회복 (Substantial Recovery)"
    elif score >= 60:
        return "중간 회복 (Moderate Recovery)"
    elif score >= 40:
        return "경미한 회복 (Mild Recovery)"
    elif score >= 20:
        return "최소 회복 (Minimal Recovery)"
    else:
        return "회복 미미 (Little to No Recovery)"

def identify_recovery_trajectory(df):
    """환자의 회복 궤적 식별 및 회복 속도 분석 - ZeroDivisionError 해결"""
    # 결과를 저장할 딕셔너리
    trajectories = {}
    
    # 날짜 형식 변환 확인
    if not pd.api.types.is_datetime64_any_dtype(df['날짜']):
        df = df.copy()
        df['날짜'] = pd.to_datetime(df['날짜'])
    
    # 각 환자별로 데이터 처리
    for patient_id, group in df.groupby('환자번호'):
        # 날짜순으로 정렬
        group = group.sort_values('날짜')
        
        # 방문 횟수가 2회 미만인 경우 건너뛰기
        if len(group) < 2:
            continue
        
        # 첫 방문과 마지막 방문 날짜 추출
        first_visit = group['날짜'].iloc[0]
        last_visit = group['날짜'].iloc[-1]
        
        # 방문 간격(일) 계산
        days_diff = (last_visit - first_visit).total_seconds() / (60*60*24)
        
        # 방문 간격이 0 이하인 경우 건너뛰기
        if days_diff <= 0:
            continue
        
        # 필요한 임상 지표 추출
        changes = {}
        
        # VAS 변화율 계산 (통증 감소)
        if 'CC_vas' in group.columns:
            first_vas = group['CC_vas'].iloc[0]
            last_vas = group['CC_vas'].iloc[-1]
            
            if pd.notna(first_vas) and pd.notna(last_vas) and first_vas > 0:
                changes['vas_daily_change'] = (last_vas - first_vas) / days_diff
                changes['vas_total_change_pct'] = (last_vas - first_vas) / first_vas * 100
        
        # CMO 변화율 계산 (개구량 증가)
        if 'CMO_before' in group.columns:
            first_cmo = group['CMO_before'].iloc[0]
            last_cmo = group['CMO_before'].iloc[-1]
            
            if pd.notna(first_cmo) and pd.notna(last_cmo) and first_cmo > 0:
                changes['cmo_daily_change'] = (last_cmo - first_cmo) / days_diff
                changes['cmo_total_change_pct'] = (last_cmo - first_cmo) / first_cmo * 100
        
        # MMO 변화율 계산 (개구량 증가)
        if 'MMO_before' in group.columns:
            first_mmo = group['MMO_before'].iloc[0]
            last_mmo = group['MMO_before'].iloc[-1]
            
            if pd.notna(first_mmo) and pd.notna(last_mmo) and first_mmo > 0:
                changes['mmo_daily_change'] = (last_mmo - first_mmo) / days_diff
                changes['mmo_total_change_pct'] = (last_mmo - first_mmo) / first_mmo * 100
        
        # 계산된 변화율이 있는 경우에만 회복 궤적 분류
        if changes:
            trajectory_type = classify_recovery_trajectory(changes)
            
            # 결과 저장
            trajectories[patient_id] = {
                'visits_count': len(group),
                'treatment_days': days_diff,
                'changes': changes,
                'trajectory_type': trajectory_type
            }
    
    return trajectories

def classify_recovery_trajectory(changes):
    """변화율을 기반으로 회복 궤적 유형 분류"""
    # 변화율 데이터가 충분하지 않은 경우
    if len(changes) < 2:
        return "데이터 불충분"
    
    # VAS 개선 여부 (통증 감소)
    vas_improving = changes.get('vas_daily_change', 0) < -0.01  # 일일 0.01 이상 감소
    
    # CMO/MMO 개선 여부 (개구량 증가)
    cmo_improving = changes.get('cmo_daily_change', 0) > 0.05   # 일일 0.05mm 이상 증가
    mmo_improving = changes.get('mmo_daily_change', 0) > 0.05
    
    # 회복 궤적 분류
    if vas_improving and (cmo_improving or mmo_improving):
        return "전반적 빠른 회복형"
    elif vas_improving:
        return "통증 우선 개선형"
    elif cmo_improving or mmo_improving:
        return "기능 우선 개선형"
    elif changes.get('vas_total_change_pct', 0) < -10 or changes.get('cmo_total_change_pct', 0) > 10:
        return "완만한 회복형"
    else:
        return "제한적 회복형"

### 임베딩 생성

#### 임베딩 및 맥락 부여 함수 정의

In [3]:
import pandas as pd
import openai
import numpy as np

def create_contextual_text(row):
    """
    환자 데이터를 임상적 맥락을 강화한 방식으로 구조화하여 텍스트 생성
    """
    # 안전하게 열 접근하는 헬퍼 함수
    def safe_access(row, key):
        if isinstance(row, dict):
            return row.get(key, None)
        else:
            try:
                return row[key] if key in row.index else None
            except:
                return None
    
    # 값 유효성 확인 헬퍼 함수
    def is_valid(value):
        return pd.notna(value) and value is not None
    
    # 최종 임상 텍스트를 담을 섹션별 컨테이너
    clinical_sections = []
    
    # 1. 주호소 및 증상 섹션 (Core Symptoms)
    symptoms_context = []
    
    # 증상 위치와 종류 통합
    cc_location = safe_access(row, 'CC_location')
    cc_pain_type = safe_access(row, 'CC_pain_type')
    
    if is_valid(cc_location) and is_valid(cc_pain_type):
        symptoms_context.append(f"주호소: 환자는 {cc_location}에 {cc_pain_type}을 호소합니다.")
    elif is_valid(cc_location):
        symptoms_context.append(f"주호소 위치: {cc_location}")
    elif is_valid(cc_pain_type):
        symptoms_context.append(f"통증 유형: {cc_pain_type}")
    
    # 턱 관련 통증과 불편감 세부 정보 추가
    cc_painUncomp_desc_jaw = safe_access(row, 'CC_painUncomp_desc_jaw')
    if is_valid(cc_painUncomp_desc_jaw):
        symptoms_context.append(f"턱 관련 통증 및 불편감: {cc_painUncomp_desc_jaw}")
    
    # 기능적 제한 정보 추가
    cc_disable_desc_jaw = safe_access(row, 'CC_disable_desc_jaw')
    if is_valid(cc_disable_desc_jaw):
        symptoms_context.append(f"턱 기능 제한: {cc_disable_desc_jaw}")
    
    
    # 근육과 관절 관련 증상 추가
    cc_muscle_joint_desc_stress = safe_access(row, 'CC_muscle_joint_desc_stress')
    if is_valid(cc_muscle_joint_desc_stress):
        symptoms_context.append(f"근육 및 관절 상태: {cc_muscle_joint_desc_stress}")
    
    # 통증 강도와 지속 기간 정보
    cc_severity = safe_access(row, 'CC_severity')
    if is_valid(cc_severity):
        symptoms_context.append(f"통증 강도(1-5): {cc_severity}")
    
    cc_vas = safe_access(row, 'CC_vas')
    if is_valid(cc_vas):
        symptoms_context.append(f"VAS 통증 점수: {cc_vas}")
    
    cc_duration = safe_access(row, 'CC_duration')
    if is_valid(cc_duration):
        symptoms_context.append(f"증상 지속 기간: {cc_duration}")
    
    # 증상 섹션을 통합하여 전체 맥락에 추가
    if symptoms_context:
        clinical_sections.append("【증상 정보】\n" + "\n".join(symptoms_context))
    
    # 2. 병력 및 습관 섹션 (History & Habits)
    history_context = []
    
    # 치과 관련 과거력
    cc_dentalHistory_desc = safe_access(row, 'CC_dentalHistory_desc')
    if is_valid(cc_dentalHistory_desc):
        history_context.append(f"치과 병력: {cc_dentalHistory_desc}")
    
    # 클리닉 방문 이력
    cc_clinic_history_desc = safe_access(row, 'CC_clinic_history_desc')
    if is_valid(cc_clinic_history_desc):
        history_context.append(f"턱관절 관련 과거 치료: {cc_clinic_history_desc}")
    
    # 생활 습관 요인
    cc_factor_habbit = safe_access(row, 'CC_factor_habbit')
    if is_valid(cc_factor_habbit):
        history_context.append(f"생활 습관 요인: {cc_factor_habbit}")
    
    # 습관 관련 상세 정보
    habit_details = []
    
    habit_type = safe_access(row, '습관_habit_type')
    if is_valid(habit_type):
        habit_details.append(f"습관 유형: {habit_type}")
    
    habit_frequency = safe_access(row, '습관_frequency')
    if is_valid(habit_frequency):
        habit_details.append(f"습관 빈도: {habit_frequency}")
    
    habit_awareness = safe_access(row, '습관_awareness')
    if is_valid(habit_awareness):
        habit_details.append(f"습관 인지 여부: {habit_awareness}")
    
    habit_improvement = safe_access(row, '습관_improvement')
    if is_valid(habit_improvement):
        habit_details.append(f"습관 개선 상태: {habit_improvement}")
    
    if habit_details:
        history_context.append("습관 상세정보: " + ", ".join(habit_details))
    
    # 병력 및 습관 섹션을 통합하여 전체 맥락에 추가
    if history_context:
        clinical_sections.append("【병력 및 습관】\n" + "\n".join(history_context))
    
    # 3. 치료 관련 섹션 (Treatment)
    treatment_context = []
    
    # 치료 계획
    cc_treat_plan = safe_access(row, 'CC_treat_plan')
    if is_valid(cc_treat_plan):
        treatment_context.append(f"치료 계획: {cc_treat_plan}")
    
    # 약물 관련 정보
    medication_details = []
    
    medication_type = safe_access(row, '약_medication_type')
    if is_valid(medication_type):
        medication_details.append(f"약물 유형: {medication_type}")
    
    medication_frequency = safe_access(row, '약_frequency')
    if is_valid(medication_frequency):
        medication_details.append(f"복용 빈도: {medication_frequency}")
    
    medication_duration = safe_access(row, '약_duration')
    if is_valid(medication_duration):
        medication_details.append(f"복용 기간: {medication_duration}")
    
    medication_compliance = safe_access(row, '약_compliance')
    if is_valid(medication_compliance):
        medication_details.append(f"복약 순응도: {medication_compliance}")
    
    if medication_details:
        treatment_context.append("약물 정보: " + ", ".join(medication_details))
    
    # 장치 관련 정보
    device_details = []
    
    device_type = safe_access(row, '장치_device_type')
    if is_valid(device_type):
        device_details.append(f"장치 유형: {device_type}")
    
    device_usage_pattern = safe_access(row, '장치_usage_pattern')
    if is_valid(device_usage_pattern):
        device_details.append(f"사용 패턴: {device_usage_pattern}")
    
    device_duration = safe_access(row, '장치_duration')
    if is_valid(device_duration):
        device_details.append(f"사용 기간: {device_duration}")
    
    device_compliance = safe_access(row, '장치_compliance')
    if is_valid(device_compliance):
        device_details.append(f"장치 순응도: {device_compliance}")
    
    if device_details:
        treatment_context.append("장치 정보: " + ", ".join(device_details))
    
    # 찜질/마사지 정보
    therapy_details = []
    
    # 찜질 정보
    hot_pack_info = []
    hot_pack_status = safe_access(row, '찜질_status')
    
    if is_valid(hot_pack_status) and hot_pack_status == 1:
        hot_pack_info.append("찜질 시행")
        
        hot_pack_frequency = safe_access(row, '찜질_frequency')
        if is_valid(hot_pack_frequency):
            hot_pack_info.append(f"빈도: {hot_pack_frequency}")
        
        hot_pack_duration = safe_access(row, '찜질_duration')
        if is_valid(hot_pack_duration):
            hot_pack_info.append(f"시간: {hot_pack_duration}분")
        
        hot_pack_method = safe_access(row, '찜질_method')
        if is_valid(hot_pack_method):
            hot_pack_info.append(f"방법: {hot_pack_method}")
    
    if hot_pack_info:
        therapy_details.append("찜질: " + ", ".join(hot_pack_info))
    
    # 마사지/스트레칭 정보
    massage_info = []
    
    massage_type = safe_access(row, '마사지, 스트레칭_type')
    if is_valid(massage_type):
        massage_info.append(f"유형: {massage_type}")
        
        massage_frequency = safe_access(row, '마사지, 스트레칭_frequency')
        if is_valid(massage_frequency):
            massage_info.append(f"빈도: {massage_frequency}")
        
        massage_duration = safe_access(row, '마사지, 스트레칭_duration')
        if is_valid(massage_duration):
            massage_info.append(f"시간: {massage_duration}분")
        
        massage_method = safe_access(row, '마사지, 스트레칭_method')
        if is_valid(massage_method):
            massage_info.append(f"방법: {massage_method}")
    
    if massage_info:
        therapy_details.append("마사지/스트레칭: " + ", ".join(massage_info))
    
    if therapy_details:
        treatment_context.append("치료 요법: " + "; ".join(therapy_details))
    
    # 치료 섹션을 통합하여 전체 맥락에 추가
    if treatment_context:
        clinical_sections.append("【치료 정보】\n" + "\n".join(treatment_context))
    
    # 4. 임상 측정 데이터 섹션 (Clinical Measurements)
    measurements_context = []
    
    # 입 벌림 관련 측정값
    opening_measurements = []
    
    cmo_before = safe_access(row, 'CMO_before')
    if is_valid(cmo_before):
        opening_measurements.append(f"편안한 개구량(처음): {cmo_before}mm")
    
    cmo_after = safe_access(row, 'CMO_after')
    if is_valid(cmo_after):
        opening_measurements.append(f"편안한 개구량(이후): {cmo_after}mm")
    
    mmo_before = safe_access(row, 'MMO_before')
    if is_valid(mmo_before):
        opening_measurements.append(f"최대 개구량(처음): {mmo_before}mm")
    
    mmo_after = safe_access(row, 'MMO_after')
    if is_valid(mmo_after):
        opening_measurements.append(f"최대 개구량(이후): {mmo_after}mm")
    
    if opening_measurements:
        measurements_context.append("개구량 측정: " + ", ".join(opening_measurements))
    
    # 턱 편위 관련 정보
    deviation_info = []
    
    deviation_pattern = safe_access(row, 'deviation_pattern_type')
    if is_valid(deviation_pattern):
        deviation_info.append(f"편위 패턴: {deviation_pattern}")
    
    deviation_direction = safe_access(row, 'deviation_direction')
    if is_valid(deviation_direction):
        deviation_info.append(f"편위 방향: {deviation_direction}")
    
    deviation_intensity = safe_access(row, 'deviation_intensity')
    if is_valid(deviation_intensity):
        deviation_info.append(f"편위 강도: {deviation_intensity}")
    
    if deviation_info:
        measurements_context.append("턱 편위: " + ", ".join(deviation_info))
    
    # 턱관절 통증 관련 정보
    pain_info = []
    
    # 촉진 시 통증
    cap_pain_intensity = safe_access(row, 'Cap.pal_Pain_Intensity')
    if is_valid(cap_pain_intensity):
        cap_pain_direction = safe_access(row, 'Cap.pal_Pain_Direction')
        cap_pain_situation = safe_access(row, 'Cap.pal_Pain_Situation')
        
        cap_pain_text = f"촉진 시 통증 강도: {cap_pain_intensity}"
        if is_valid(cap_pain_direction):
            cap_pain_text += f", 방향: {cap_pain_direction}"
        if is_valid(cap_pain_situation):
            cap_pain_text += f", 상황: {cap_pain_situation}"
        
        pain_info.append(cap_pain_text)
    
    # 운동 시 통증
    m_pain_intensity = safe_access(row, 'M.pal_Pain_Intensity')
    if is_valid(m_pain_intensity):
        m_pain_direction = safe_access(row, 'M.pal_Pain_Direction')
        m_pain_situation = safe_access(row, 'M.pal_Pain_Situation')
        
        m_pain_text = f"운동 시 통증 강도: {m_pain_intensity}"
        if is_valid(m_pain_direction):
            m_pain_text += f", 방향: {m_pain_direction}"
        if is_valid(m_pain_situation):
            m_pain_text += f", 상황: {m_pain_situation}"
        
        pain_info.append(m_pain_text)
    
    if pain_info:
        measurements_context.append("턱관절 통증: " + ", ".join(pain_info))
    
    # 턱관절 소리 관련 정보
    noise_info = []
    
    noise_code = safe_access(row, 'Noise_Code')
    if is_valid(noise_code) and noise_code != 'No-Noise':
        noise_info.append(f"소리 유형: {noise_code}")
        
        noise_direction = safe_access(row, 'Noise_Direction')
        if is_valid(noise_direction):
            noise_info.append(f"방향: {noise_direction}")
        
        noise_intensity = safe_access(row, 'Noise_Intensity')
        if is_valid(noise_intensity):
            noise_info.append(f"강도: {noise_intensity}")
        
        noise_situation = safe_access(row, 'Noise_Situation')
        if is_valid(noise_situation):
            noise_info.append(f"발생 상황: {noise_situation}")
    
    if noise_info:
        measurements_context.append("턱관절 소리: " + ", ".join(noise_info))
    
    # 교합 관련 정보
    occlusion_info = []
    
    occlusion_lt_number = safe_access(row, 'Occlusion_lt_number')
    occlusion_rt_number = safe_access(row, 'Occlusion_rt_number')
    occlusion_lt_intensity = safe_access(row, 'Occlusion_lt_Intensity')
    occlusion_rt_intensity = safe_access(row, 'Occlusion_rt_Intensity')
    
    if is_valid(occlusion_lt_number) or is_valid(occlusion_rt_number):
        lt_info = f"왼쪽 교합 치아 수: {occlusion_lt_number if is_valid(occlusion_lt_number) else '정보 없음'}"
        rt_info = f"오른쪽 교합 치아 수: {occlusion_rt_number if is_valid(occlusion_rt_number) else '정보 없음'}"
        
        lt_intensity = f"왼쪽 교합 강도: {occlusion_lt_intensity if is_valid(occlusion_lt_intensity) else '정보 없음'}"
        rt_intensity = f"오른쪽 교합 강도: {occlusion_rt_intensity if is_valid(occlusion_rt_intensity) else '정보 없음'}"
        
        occlusion_info.extend([lt_info, rt_info, lt_intensity, rt_intensity])
    
    oj_value = safe_access(row, 'oj')
    if is_valid(oj_value):
        occlusion_info.append(f"수평피개량: {oj_value}mm")
    
    ob_value = safe_access(row, 'ob')
    if is_valid(ob_value):
        occlusion_info.append(f"수직피개량: {ob_value}mm")
    
    # 정중선 편위 정보
    midline_shift_jaw = safe_access(row, 'Midline_Shift_Jaw')
    if is_valid(midline_shift_jaw):
        midline_info = f"정중선 편위 턱: {midline_shift_jaw}"
        
        midline_dir_x = safe_access(row, 'Midline_Shift_Direction_x')
        if is_valid(midline_dir_x):
            midline_info += f", X축 방향: {midline_dir_x}"
        
        midline_dir_y = safe_access(row, 'Midline_Shift_Direction_y')
        if is_valid(midline_dir_y):
            midline_info += f", Y축 방향: {midline_dir_y}"
        
        midline_amount = safe_access(row, 'Midline_Shift_Amount')
        if is_valid(midline_amount):
            midline_info += f", 편위량: {midline_amount}mm"
        
        occlusion_info.append(midline_info)
    
    # CR-CO 관련 정보
    crco_dir_x = safe_access(row, 'CRCO_Direction_x')
    if is_valid(crco_dir_x):
        crco_info = f"CRCO X축 방향: {crco_dir_x}"
        
        crco_dir_y = safe_access(row, 'CRCO_Direction_y')
        if is_valid(crco_dir_y):
            crco_info += f", Y축 방향: {crco_dir_y}"
        
        crco_amount = safe_access(row, 'CRCO_Amount')
        if is_valid(crco_amount):
            crco_info += f", 양: {crco_amount}"
        
        occlusion_info.append(crco_info)
    
    # 혀 압흔 정보
    tongue_ridging = safe_access(row, 'Tongue_ridging_Intensity')
    if is_valid(tongue_ridging):
        occlusion_info.append(f"혀 압흔 강도: {tongue_ridging}")
    # 혀 압흔 정보
    mucosal_ridging = safe_access(row, 'Mucosal_ridging_Intensity')
    if is_valid(mucosal_ridging):
        occlusion_info.append(f"점막 압흔 강도: {mucosal_ridging}")
    
    # 우측 및 좌측 측방 이동량
    rt_before = safe_access(row, 'Rt_before')
    rt_after = safe_access(row, 'Rt_after')
    lt_before = safe_access(row, 'Lt_before')
    lt_after = safe_access(row, 'Lt_after')
    
    lateral_movement_info = []
    if is_valid(rt_before):
        lateral_movement_info.append(f"우측 측방 이동량(처음): {rt_before}mm")
    if is_valid(rt_after):
        lateral_movement_info.append(f"우측 측방 이동량(이후): {rt_after}mm")
    if is_valid(lt_before):
        lateral_movement_info.append(f"좌측 측방 이동량(처음): {lt_before}mm")
    if is_valid(lt_after):
        lateral_movement_info.append(f"좌측 측방 이동량(이후): {lt_after}mm")
    
    if lateral_movement_info:
        occlusion_info.extend(lateral_movement_info)
    
    if occlusion_info:
        measurements_context.append("교합 및 턱 운동 상태: " + ", ".join(occlusion_info))
    
    # 임상 측정값 섹션을 통합하여 전체 맥락에 추가
    if measurements_context:
        clinical_sections.append("【임상 측정 데이터】\n" + "\n".join(measurements_context))
    
    # 5. 결과/치료 효과 섹션 (Outcomes)
    outcomes_context = []
    
    # 치료 효과 관련 정보
    next_visit_days = safe_access(row, 'Next_Visit_Days')
    if is_valid(next_visit_days):
        outcomes_context.append(f"다음 방문 예정일: {next_visit_days}일 후")
    
    # VAS 점수 변화 (이 부분은 위의 CC_vas와 중복될 수 있지만, 결과 섹션에도 포함)
    cc_vas_outcome = safe_access(row, 'CC_vas')
    if is_valid(cc_vas_outcome):
        outcomes_context.append(f"현재 통증 점수(VAS): {cc_vas_outcome}")
    
    # 결과 섹션을 통합하여 전체 맥락에 추가
    if outcomes_context:
        clinical_sections.append("【치료 결과】\n" + "\n".join(outcomes_context))
    
    # 모든 섹션을 통합하여 하나의 맥락화된 텍스트 생성
    contextual_text = "\n\n".join(clinical_sections)
    
    # 환자 ID와 날짜 정보를 헤더로 추가
    patient_id = safe_access(row, '환자번호')
    visit_date = safe_access(row, '날짜')
    
    header = f"환자번호: {patient_id if is_valid(patient_id) else '정보 없음'}, 방문일: {visit_date if is_valid(visit_date) else '정보 없음'}"
    
    # 최종 맥락화된 텍스트
    final_text = f"{header}\n\n{contextual_text}"
    
    return final_text

def enhance_clinical_relationships(row): #-> 너무 뇌피셜임. 일단 누락.
    """임상적으로 관련된 필드들 간의 관계를 강화"""
    relationships = []
    
    # 증상과 치료의 관계
    if pd.notna(row['CC_pain_type']) and pd.notna(row['CC_treat_plan']):
        relationships.append(f"{row['CC_pain_type']}에 대한 치료 계획: {row['CC_treat_plan']}")
    
    # 습관과 증상의 관계
    if pd.notna(row['습관_habit_type']) and pd.notna(row['CC_muscle_joint_desc_stress']):
        relationships.append(f"습관 '{row['습관_habit_type']}'과 근육 상태 '{row['CC_muscle_joint_desc_stress']}'의 연관성")
    
    # 치료와 결과의 관계
    if pd.notna(row['장치_device_type']) and pd.notna(row['CC_vas']):
        relationships.append(f"{row['장치_device_type']} 장치 사용 후 통증 점수: {row['CC_vas']}")
    
    return relationships

def identify_patient_patterns(row):
    """환자 데이터에서 중요 패턴 식별"""
    patterns = []
    
    # 통증 위치와 종류 패턴
    if pd.notna(row['CC_location']) and pd.notna(row['CC_pain_type']):
        patterns.append(f"패턴-위치통증: {row['CC_location']}_{row['CC_pain_type']}")
    
    # 개구량 변화 패턴
    if pd.notna(row['MMO_before']) and pd.notna(row['MMO_after']):
        try:
            change = float(row['MMO_after']) - float(row['MMO_before'])
            if change > 3:
                patterns.append("패턴-개구량: 뚜렷한_개선")
            elif change > 0:
                patterns.append("패턴-개구량: 경미한_개선")
            elif change < -3:
                patterns.append("패턴-개구량: 악화")
            else:
                patterns.append("패턴-개구량: 유지")
        except:
            pass
    
    # 습관과 순응도 패턴
    if pd.notna(row['습관_awareness']) and pd.notna(row['장치_compliance']):
        patterns.append(f"패턴-순응도: 습관인지_{row['습관_awareness']}_장치순응도_{row['장치_compliance']}")
    
    return patterns

def apply_section_weights(text, section_weights):
    """섹션별 가중치를 적용한 텍스트 생성 - 특수 마커 사용"""
    sections = text.split("\n\n")
    weighted_sections = []
    
    for section in sections:
        for key, weight in section_weights.items():
            if key in section:
                # 섹션 시작 부분에 가중치 마커 추가
                weighted_section = f"<WEIGHT={weight}> {section}"
                weighted_sections.append(weighted_section)
                break
        else:
            weighted_sections.append(section)
    
    return "\n\n".join(weighted_sections)


#### 임베딩 함수

In [4]:
def create_hybrid_contextual_embedding(df, api_key):
    """환자 데이터의 하이브리드 맥락 임베딩 생성 - 시계열 고려"""
    
    # OpenAI 클라이언트 초기화 (v1.0.0 이상)
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
    except ImportError as e:
        print(f"OpenAI 라이브러리 가져오기 실패: {e}")
        print("pip install openai 명령으로 OpenAI 라이브러리를 설치해주세요.")
        return []
    
    # 환자 ID별로 방문 데이터 그룹화
    patient_groups = df.sort_values('날짜').groupby('환자번호')
    
    # 임베딩 저장할 리스트
    embeddings = []

    # 섹션 가중치 정의 (원본과 동일하게 유지)
    section_weights = {
        "【증상 정보】": 2,  # 증상 정보는 2배 중요
        "【치료 결과】": 2,  # 치료 결과도 2배 중요
        "【병력 및 습관】": 1.5  # 병력 및 습관은 1.5배 중요
    }
    
    for patient_id, patient_visits in patient_groups:
        # 환자별 방문이 1회인 경우와 여러 회인 경우 구분 처리
        if len(patient_visits) == 1:
            # 단일 방문 처리 (기존 방식과 동일)
            row = patient_visits.iloc[0]
            row_dict = row.to_dict()
            
            # 1. 맥락화된 텍스트 생성
            clinical_text = create_contextual_text(row_dict)
            
            # 2. 임상적 관계 강화
            relationships = enhance_clinical_relationships(row_dict)
            if relationships:
                clinical_text += "\n\n【임상적 관계】\n" + "\n".join(relationships)
            
            # 3. 환자 패턴 식별 및 추가
            patterns = identify_patient_patterns(row_dict)
            if patterns:
                clinical_text += "\n\n【패턴 정보】\n" + "\n".join(patterns)
            
            # 4. 섹션 가중치 적용
            weighted_text = apply_section_weights(clinical_text, section_weights)
            visit_text = weighted_text  # 단일 방문용 텍스트
            
        else:
            # 여러 방문 처리 (시계열 통합)
            visit_texts = []
            
            # 각 방문별로 맥락화된 텍스트 생성
            for idx, row in patient_visits.iterrows():
                row_dict = row.to_dict()
                
                # 방문 날짜 포맷팅
                visit_date = row_dict['날짜']
                if hasattr(visit_date, 'strftime'):
                    visit_date = visit_date.strftime('%Y-%m-%d')
                
                # 각 방문에 대한 맥락화 텍스트 생성 (기존 로직 재사용)
                clinical_text = create_contextual_text(row_dict)
                
                # 임상적 관계 및 패턴 정보 추가
                relationships = enhance_clinical_relationships(row_dict)
                if relationships:
                    clinical_text += "\n\n【임상적 관계】\n" + "\n".join(relationships)
                
                patterns = identify_patient_patterns(row_dict)
                if patterns:
                    clinical_text += "\n\n【패턴 정보】\n" + "\n".join(patterns)
                
                # 섹션 가중치 적용
                weighted_text = apply_section_weights(clinical_text, section_weights)
                
                # 방문일자 헤더와 함께 저장
                visit_text = f"【방문일: {visit_date}】\n{weighted_text}"
                visit_texts.append(visit_text)
            
            # 시간 순서를 보존하여 모든 방문 정보 통합
            visit_text = "\n\n===== 다음 방문 =====\n\n".join(visit_texts)
            
            # 환자 시계열 요약 추가
            if len(patient_visits) >= 2:
                first_visit = patient_visits.iloc[0]
                last_visit = patient_visits.iloc[-1]
                
                # 주요 수치의 변화 요약 계산
                summary_lines = ["【환자 시계열 요약】"]
                
                for measure in ['CC_vas', 'CMO_before', 'MMO_before']:
                    if measure in patient_visits.columns:
                        first_val = first_visit.get(measure)
                        last_val = last_visit.get(measure)
                        
                        if pd.notna(first_val) and pd.notna(last_val):
                            change = last_val - first_val
                            change_pct = (change / first_val * 100) if first_val != 0 else float('inf')
                            
                            summary_lines.append(f"{measure} 변화: {first_val:.2f} → {last_val:.2f} (변화량: {change:.2f}, 변화율: {change_pct:.1f}%)")
                
                visit_time_span = (last_visit['날짜'] - first_visit['날짜']).days
                summary_lines.append(f"치료 기간: {visit_time_span}일, 방문 횟수: {len(patient_visits)}회")
                
                # 요약 정보 추가
                visit_text = "\n".join(summary_lines) + "\n\n" + visit_text
        
        # 맥락화된 텍스트 샘플 출력 (테스트용)
        print(f"환자 {patient_id}의 맥락화된 텍스트 샘플:\n{visit_text[:500]}...\n")
        
        # 임베딩 생성 시도 - OpenAI API 호출
        try:
            response = client.embeddings.create(
                model="text-embedding-ada-002",
                input=visit_text
            )
            embedding = response.data[0].embedding
            
            # 마지막 방문에 대한 정보 취득
            last_row = patient_visits.iloc[-1]
            
            # 시계열 특성 계산
            time_features = {}
            if len(patient_visits) > 1:
                # 방문 간격 통계
                visit_dates = pd.to_datetime(patient_visits['날짜'])
                visit_intervals = [(visit_dates.iloc[i] - visit_dates.iloc[i-1]).days 
                                   for i in range(1, len(visit_dates))]
                
                time_features['방문횟수'] = len(patient_visits)
                time_features['총기간_일'] = (visit_dates.iloc[-1] - visit_dates.iloc[0]).days
                time_features['평균방문간격_일'] = np.mean(visit_intervals) if visit_intervals else 0
                
                # 주요 지표의 변화 추세
                for measure in ['CC_vas', 'CMO_before', 'MMO_before']:
                    if measure in patient_visits.columns:
                        values = patient_visits[measure].dropna()
                        if len(values) >= 2:
                            # 선형 추세 분석
                            x = np.arange(len(values))
                            if len(x) > 0 and len(values) > 0:
                                slope, intercept = np.polyfit(x, values, 1)
                                time_features[f'{measure}_추세_기울기'] = slope
            
            # 결과 저장 - 임베딩, 환자 ID, 방문 날짜, 수치값, 시계열 특성 포함
            embeddings.append({
                'patient_id': patient_id,
                'visit_date': last_row.get('날짜', None),
                'embedding': embedding,
                'visit_count': len(patient_visits),
                'time_features': time_features,
                'vas': last_row.get('CC_vas', None) if pd.notna(last_row.get('CC_vas', None)) else None,
                'cmo': last_row.get('CMO_before', None) if pd.notna(last_row.get('CMO_before', None)) else None,
                'mmo': last_row.get('MMO_before', None) if pd.notna(last_row.get('MMO_before', None)) else None
            })
            
            print(f"환자 {patient_id}의 임베딩 생성 완료 (차원: {len(embedding)})")
            
        except Exception as e:
            print(f"환자 {patient_id}의 임베딩 생성 중 오류 발생: {str(e)}")
    
    return embeddings

#### 임베딩 API 실행 및 JSON 저장

In [6]:
# 메인 함수 - OpenAI API v1.0.0+ 호환
# 메인 함수 - OpenAI API v1.0.0+ 호환
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# 사용
def do_create_embeddings(df, batch_size=500):
    save_dir = '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings'
    # 적은 수의 샘플만 테스트 (전체 데이터셋을 처리하려면 이 부분 주석 처리)
    
    print(f"총 {len(df)}개의 환자 데이터에 대해 임베딩을 생성합니다...")
    
    # 배치 크기 설정
    batch_size = batch_size
    total_rows = len(df)
    
    # 배치 단위로 처리
    for i in tqdm(range(0, total_rows, batch_size), desc="배치 처리 중"):
        # 현재 배치의 끝 인덱스 계산
        end_idx = min(i + batch_size, total_rows)
        
        # 현재 배치 추출
        batch_df = df.iloc[i:end_idx]
        
        # 임베딩 생성
        batch_embeddings = create_hybrid_contextual_embedding(batch_df, api_key)
        
        # 배치별 파일명 설정
        # batch_filename = f'patient_embeddings_batch_{i//batch_size+1}.json'
        batch_filename = os.path.join(save_dir, f'patient_embeddings_batch_{i//batch_size+1}.json')
        # 임베딩 저장
        save_embeddings_to_json(batch_embeddings, batch_filename)
        
        print(f"배치 {i//batch_size+1}: {len(batch_embeddings)}개 환자 데이터의 임베딩이 '{batch_filename}'에 저장되었습니다.")
    
    print(f"총 {total_rows}개 환자 데이터의 임베딩이 생성되어 {(total_rows-1)//batch_size+1}개의 배치 파일로 저장되었습니다.")
    
    # 첫 번째 배치 파일에서 임베딩 차원 정보 읽기
    try:
        with open(f'patient_embeddings_batch_1.json', 'r', encoding='utf-8') as f:
            first_batch = json.load(f)
            if first_batch:
                print("임베딩 차원:", len(first_batch[0]['embedding']) if 'embedding' in first_batch[0] else "없음")
    except Exception as e:
        print( os.path.join(save_dir,f"임베딩 차원 정보를 읽는 중 오류 발생: {e}"))


# 사용
def save_embeddings_to_json(embeddings, filename):
    """임베딩을 JSON 파일로 저장하는 함수"""
    # JSON 직렬화를 위해 NumPy 배열과 Timestamp 객체 처리
    serializable_embeddings = []
    for item in embeddings:
        serializable_item = item.copy()
        
        # 임베딩 벡터가 NumPy 배열인 경우 리스트로 변환
        if 'embedding' in serializable_item and hasattr(serializable_item['embedding'], 'tolist'):
            serializable_item['embedding'] = serializable_item['embedding'].tolist()
        
        # Timestamp 객체가 있는 경우 문자열로 변환
        if 'visit_date' in serializable_item and hasattr(serializable_item['visit_date'], 'strftime'):
            serializable_item['visit_date'] = serializable_item['visit_date'].strftime('%Y-%m-%d')
        
        serializable_embeddings.append(serializable_item)
    
    # 커스텀 JSON 인코더 사용
    class CustomJSONEncoder(json.JSONEncoder):
        def default(self, obj):
            # pandas Timestamp 객체 처리
            if hasattr(obj, 'strftime'):
                return obj.strftime('%Y-%m-%d')
            # NumPy 배열 처리
            if hasattr(obj, 'tolist'):
                return obj.tolist()
            return super().default(obj)
    
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(serializable_embeddings, f, cls=CustomJSONEncoder, ensure_ascii=False, indent=2)



총 213개의 환자 데이터에 대해 임베딩을 생성합니다...


배치 처리 중:   0%|          | 0/11 [00:00<?, ?it/s]

환자 2202-62의 맥락화된 텍스트 샘플:
【환자 시계열 요약】
CC_vas 변화: 5.00 → 3.00 (변화량: -2.00, 변화율: -40.0%)
CMO_before 변화: 18.00 → 47.00 (변화량: 29.00, 변화율: 161.1%)
MMO_before 변화: 24.00 → 47.00 (변화량: 23.00, 변화율: 95.8%)
치료 기간: 432일, 방문 횟수: 20회

【방문일: 2022-02-16】
환자번호: 2202-62, 방문일: 2022-02-16 00:00:00

<WEIGHT=2> 【증상 정보】
주호소: 환자는 입에 통증을 호소합니다.
턱 관련 통증 및 불편감: Unknown
턱 기능 제한: 개구 제한
근육 및 관절 상태: 치아 악물기
통증 강도(1-5): 5.0
VAS 통증 점수: 5.0
증상 지속 기간: 10.0

<WEIGHT=1.5> 【병력 및 습관】
치과 병력: Unknown
턱관절 관련 과거 치료: Unknown
생활 습관 요인: Unknown
습관 상세정보: 습관 유형: 치아 접촉 ...



배치 처리 중:   9%|▉         | 1/11 [00:01<00:15,  1.59s/it]

환자 2202-62의 임베딩 생성 중 오류 발생: Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens, however you requested 24591 tokens (24591 in your prompt; 0 for the completion). Please reduce your prompt; or completion length.", 'type': 'invalid_request_error', 'param': None, 'code': None}}
배치 1: 0개 환자 데이터의 임베딩이 '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/patient_embeddings_batch_1.json'에 저장되었습니다.
환자 2202-62의 맥락화된 텍스트 샘플:
【환자 시계열 요약】
CC_vas 변화: 3.00 → 4.00 (변화량: 1.00, 변화율: 33.3%)
CMO_before 변화: 52.00 → 49.00 (변화량: -3.00, 변화율: -5.8%)
MMO_before 변화: 52.00 → 49.00 (변화량: -3.00, 변화율: -5.8%)
치료 기간: 297일, 방문 횟수: 8회

【방문일: 2023-06-02】
환자번호: 2202-62, 방문일: 2023-06-02 00:00:00

<WEIGHT=2> 【증상 정보】
주호소: 환자는 왼쪽 턱에 뻐근함을 호소합니다.
턱 관련 통증 및 불편감: Unknown
턱 기능 제한: 제한된 개구
근육 및 관절 상태: Unknown
통증 강도(1-5): 1.0
VAS 통증 점수: 3.0
증상 지속 기간: 10.0

<WEIGHT=1.5> 【병력 및 습관】
치과 병력: Unknown
턱관절 관련 과거 치료: Unknown
생활 습관 요인: Unknown
습관 상세정보: 습관 유형: 치아 갈...

환자 2202-62의 임베딩 생성 중 오류

...

환자 2304-05의 임베딩 생성 완료 (차원: 1536)
배치 9: 1개 환자 데이터의 임베딩이 '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/patient_embeddings_batch_9.json'에 저장되었습니다.
환자 2304-05의 맥락화된 텍스트 샘플:
【환자 시계열 요약】
CC_vas 변화: 0.00 → 2.00 (변화량: 2.00, 변화율: inf%)
CMO_before 변화: 40.00 → 37.00 (변화량: -3.00, 변화율: -7.5%)
MMO_before 변화: 45.00 → 40.00 (변화량: -5.00, 변화율: -11.1%)
치료 기간: 311일, 방문 횟수: 11회

【방문일: 2023-05-09】
환자번호: 2304-05, 방문일: 2023-05-09 00:00:00

<WEIGHT=2> 【증상 정보】
주호소: 환자는 양쪽 귀앞에 뻐근함을 호소합니다.
턱 관련 통증 및 불편감: Unknown
턱 기능 제한: Unknown
근육 및 관절 상태: 근육 긴장
통증 강도(1-5): 3.0
VAS 통증 점수: 0.0
증상 지속 기간: 10.0

<WEIGHT=1.5> 【병력 및 습관】
치과 병력: Unknown
턱관절 관련 과거 치료: Unknown
생활 습관 요인: 딱딱한 음식 섭취, 수면 중 치아 갈기
습관 상세...

환자 2304-05의 임베딩 생성 중 오류 발생: Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens, however you requested 13198 tokens (13198 in your prompt; 0 for the completion). Please reduce your prompt; or completion length.", 'type': 'invalid_request_error', 'param': None, 'code

배치 처리 중:  91%|█████████ | 10/11 [00:19<00:01,  1.86s/it]

환자 2304-90의 임베딩 생성 완료 (차원: 1536)
배치 10: 1개 환자 데이터의 임베딩이 '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/patient_embeddings_batch_10.json'에 저장되었습니다.
환자 2304-90의 맥락화된 텍스트 샘플:
【환자 시계열 요약】
CC_vas 변화: 3.00 → 3.00 (변화량: 0.00, 변화율: 0.0%)
CMO_before 변화: 50.00 → 50.00 (변화량: 0.00, 변화율: 0.0%)
MMO_before 변화: 50.00 → 50.00 (변화량: 0.00, 변화율: 0.0%)
치료 기간: 105일, 방문 횟수: 3회

【방문일: 2023-07-08】
환자번호: 2304-90, 방문일: 2023-07-08 00:00:00

<WEIGHT=2> 【증상 정보】
주호소: 환자는 오른쪽 턱에 통증을 호소합니다.
턱 관련 통증 및 불편감: 턱 소리 여전(한두번정도 크게났어요)
턱 기능 제한: 입벌릴때 어긋나는 느낌 못느꼈고
근육 및 관절 상태: 마사지스트레칭 한번했어요
통증 강도(1-5): 4.0
VAS 통증 점수: 3.0
증상 지속 기간: 10.0

<WEIGHT=1.5> 【병력 및 습관】
치과 병력: Unknown
턱관절 관련 과거 치료: 3개월 전 증상 시작, 과거 외상 이력...

환자 2304-90의 임베딩 생성 완료 (차원: 1536)
환자 2306-90의 맥락화된 텍스트 샘플:
【환자 시계열 요약】
CC_vas 변화: 6.00 → 3.00 (변화량: -3.00, 변화율: -50.0%)
CMO_before 변화: 30.00 → 52.00 (변화량: 22.00, 변화율: 73.3%)
MMO_before 변화: 55.00 → 52.00 (변화량: -3.00, 변화율: -5.5%)
치료 기간: 272일, 방문 횟수: 6회

【방문일: 2023-06-24】
환자번호: 2306-90, 방문일: 2023-06-24 00:0

배치 처리 중: 100%|██████████| 11/11 [00:21<00:00,  1.97s/it]

환자 2308-293의 임베딩 생성 완료 (차원: 1536)
배치 11: 3개 환자 데이터의 임베딩이 '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/patient_embeddings_batch_11.json'에 저장되었습니다.
총 213개 환자 데이터의 임베딩이 생성되어 11개의 배치 파일로 저장되었습니다.
/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings/임베딩 차원 정보를 읽는 중 오류 발생: [Errno 2] No such file or directory: 'patient_embeddings_batch_1.json'


## 모델

### 시계열 피쳐 처리

In [7]:
def extract_time_series_features(patient_visits):
    """환자 방문 시계열에서 유용한 특성 추출"""
    features = {}
    
    try:
        # 기본 방문 통계
        features['방문횟수'] = len(patient_visits)
        
        # 날짜 처리
        visit_dates = pd.to_datetime(patient_visits['날짜'])
        if len(visit_dates) > 1:
            features['총기간_일'] = (visit_dates.iloc[-1] - visit_dates.iloc[0]).days
            
            # 방문 간격
            visit_intervals = [(visit_dates.iloc[i] - visit_dates.iloc[i-1]).days 
                               for i in range(1, len(visit_dates))]
            features['평균방문간격_일'] = np.mean(visit_intervals) if visit_intervals else 0
        
        # 주요 측정값들에 대한 시계열 특성 추출
        for col in ['CC_vas', 'CMO_before', 'MMO_before']:
            if col in patient_visits.columns:
                values = patient_visits[col].dropna()
                if len(values) >= 2:
                    # 기본 통계 
                    features[f'{col}_평균'] = float(values.mean())
                    features[f'{col}_표준편차'] = float(values.std())
                    features[f'{col}_최소'] = float(values.min())
                    features[f'{col}_최대'] = float(values.max())
                    
                    # 선형 추세 분석
                    x = np.arange(len(values))
                    slope, intercept = np.polyfit(x, values, 1)
                    features[f'{col}_추세_기울기'] = float(slope)
                    features[f'{col}_추세_절편'] = float(intercept)
                    
                    # 최근 경향 (마지막 3회 방문 또는 가능한 만큼)
                    recent_count = min(3, len(values))
                    if recent_count >= 2:
                        recent_values = values.iloc[-recent_count:]
                        recent_x = np.arange(recent_count)
                        recent_slope, recent_intercept = np.polyfit(recent_x, recent_values, 1)
                        features[f'{col}_최근추세_기울기'] = float(recent_slope)
                    
                    # 첫 방문-마지막 방문 변화량
                    features[f'{col}_첫방문값'] = float(values.iloc[0])
                    features[f'{col}_마지막방문값'] = float(values.iloc[-1])
                    features[f'{col}_총변화량'] = float(values.iloc[-1] - values.iloc[0])
                    
                    # 비선형성 점수 (순차적 변화량의 방향 변경 횟수)
                    if len(values) >= 3:
                        diffs = np.diff(values)
                        sign_changes = sum(1 for i in range(len(diffs)-1) if diffs[i] * diffs[i+1] < 0)
                        features[f'{col}_비선형성_점수'] = sign_changes
    except Exception as e:
        print(f"시계열 특성 추출 중 오류: {e}")
    
    return features

In [8]:
    
def classify_trajectory_type(slope, r_value, nonlinearity_metrics, measure_col):
    """회복 궤적의 유형 분류"""
    # 측정값에 따른 개선 방향 결정
    improving_direction = -1 if measure_col == 'CC_vas' else 1  # 통증은 감소가, 개구량은 증가가 개선
    
    # 비선형성 점수 계산
    nonlinearity_score = 0
    if 'sign_changes' in nonlinearity_metrics:
        nonlinearity_score += nonlinearity_metrics['sign_changes'] * 0.5
    if 'ma_diff_mean' in nonlinearity_metrics:
        nonlinearity_score += nonlinearity_metrics['ma_diff_mean'] * 5.0
    
    # 궤적 유형 분류
    if abs(r_value) < 0.3:  # 선형성이 매우 낮음
        return "불규칙형 (Irregular)"
    elif nonlinearity_score > 1.0:  # 비선형성이 높음
        if slope * improving_direction > 0:
            return "복합 개선형 (Complex Improvement)"
        else:
            return "복합 악화형 (Complex Deterioration)"
    else:  # 선형성이 높음
        if slope * improving_direction > 0:
            if abs(slope) > 0.1:
                return "빠른 선형 개선형 (Rapid Linear Improvement)"
            else:
                return "느린 선형 개선형 (Slow Linear Improvement)"
        else:
            if abs(slope) > 0.1:
                return "빠른 선형 악화형 (Rapid Linear Deterioration)"
            else:
                return "느린 선형 악화형 (Slow Linear Deterioration)"

In [9]:
# 사용
def model_recovery_trajectory(patient_visits, measure_col='CC_vas'):
    """환자의 회복 추세선을 모델링"""
    import numpy as np
    from scipy import stats
    
    try:
        # 날짜 및 측정값 추출
        dates = pd.to_datetime(patient_visits['날짜'])
        values = patient_visits[measure_col].dropna().values
        
        # 측정값이 부족한 경우
        if len(values) < 2:
            return None
        
        # 날짜를 숫자형 일수로 변환 (첫 방문일 기준)
        start_date = dates.min()
        days = [(date - start_date).days for date in dates]
        days = [d for d, v in zip(days, patient_visits[measure_col]) if not pd.isna(v)]
        
        # 결측치 제거 후 값 필터링
        values = [v for v in values if not pd.isna(v)]
        
        # 최소 2개 이상의 유효한 측정값이 있어야 모델링 가능
        if len(days) < 2 or len(values) < 2:
            return None
            
        # 선형 회귀
        slope, intercept, r_value, p_value, std_err = stats.linregress(days, values)
        
        # 목표값까지 예상 소요 일수 계산 (measure_col에 따라 목표값 설정)
        if measure_col == 'CC_vas':
            target_value = 0  # 통증 완전 소실
        elif measure_col == 'CMO_before':
            target_value = 43  # 정상 개구량
        else:
            target_value = None
        
        # 목표까지 예상 소요 일수 (기울기가 유의미한 경우만)
        predicted_days = None
        if abs(slope) > 0.01 and target_value is not None:
            # 마지막 측정값
            last_value = values[-1]
            # 현재 추세대로라면 목표까지 며칠이 걸릴지 계산
            if (target_value > last_value and slope > 0) or (target_value < last_value and slope < 0):
                predicted_days = abs((target_value - last_value) / slope)
        
        # 비선형 패턴 탐지
        nonlinearity_metrics = {}
        if len(days) >= 3:
            # 연속 차분의 부호 변화 횟수
            diff_signs = np.sign(np.diff(values))
            sign_changes = np.sum(np.abs(np.diff(diff_signs)) > 0)
            nonlinearity_metrics['sign_changes'] = int(sign_changes)
            
            # 이동평균과 실제값의 차이 (윈도우 크기: 데이터 길이의 1/3)
            window_size = max(2, len(days) // 3)
            if len(values) >= window_size:
                moving_avgs = []
                for i in range(len(values) - window_size + 1):
                    moving_avgs.append(np.mean(values[i:i+window_size]))
                
                # 이동평균과 실제값의 차이
                ma_diff = []
                for i in range(len(moving_avgs)):
                    ma_diff.append(abs(values[i+window_size-1] - moving_avgs[i]))
                
                nonlinearity_metrics['ma_diff_mean'] = float(np.mean(ma_diff))
        
        # 궤적 유형 분류
        trajectory_type = classify_trajectory_type(slope, r_value, nonlinearity_metrics, measure_col)
        
        return {
            'slope': float(slope),
            'intercept': float(intercept),
            'r_squared': float(r_value**2),
            'p_value': float(p_value),
            'std_err': float(std_err),
            'nonlinearity_metrics': nonlinearity_metrics,
            'predicted_days_to_target': float(predicted_days) if predicted_days is not None else None,
            'trajectory_type': trajectory_type
        }
    except Exception as e:
        print(f"회복 궤적 모델링 중 오류: {str(e)}")
        return None

In [10]:
### 사용
def extract_patient_progress(df):
    """환자별 진행 상태를 전체 방문 데이터로부터 추출"""
    # 날짜 형식 변환 확인
    if not pd.api.types.is_datetime64_any_dtype(df['날짜']):
        df = df.copy()
        df['날짜'] = pd.to_datetime(df['날짜'])
    
    # 환자별 모든 방문 데이터 분석
    patient_progress = {}
    
    # 환자별로 데이터 분석
    for patient_id, patient_visits in df.groupby('환자번호'):
        # 방문 데이터 시간순 정렬
        patient_visits = patient_visits.sort_values('날짜')
        
        # 첫 방문과 마지막 방문
        first_visit = patient_visits.iloc[0].to_dict()
        last_visit = patient_visits.iloc[-1].to_dict()
        
        # 기본 진행 정보
        progress_info = {
            '환자번호': patient_id,
            '첫방문일': patient_visits['날짜'].iloc[0],
            '마지막방문일': patient_visits['날짜'].iloc[-1],
            '총방문횟수': len(patient_visits),
            '치료기간': (patient_visits['날짜'].iloc[-1] - patient_visits['날짜'].iloc[0]).days
        }
        
        # 주요 측정값별 진행 상태 분석
        for measure in ['CC_vas', 'CMO_before', 'MMO_before', 'dif_MMO_CMO']:
            if measure in patient_visits.columns:
                # 값이 유효한 방문만 추출
                valid_visits = patient_visits[~pd.isna(patient_visits[measure])]
                
                if len(valid_visits) >= 2:
                    # 첫 방문과 마지막 방문 값
                    first_value = valid_visits[measure].iloc[0]
                    last_value = valid_visits[measure].iloc[-1]
                    
                    # 진행 상태 저장
                    progress_info[f'첫방문_{measure}'] = first_value
                    progress_info[f'마지막방문_{measure}'] = last_value
                    
                    # 중간 방문들의 값들
                    if len(valid_visits) > 2:
                        middle_values = valid_visits[measure].iloc[1:-1].tolist()
                        progress_info[f'{measure}_중간값'] = middle_values
                    
                    # 변화량 및 성장률
                    if pd.notna(first_value) and pd.notna(last_value) and first_value != 0:
                        change = last_value - first_value
                        growth_rate = change / first_value * 100
                        
                        progress_info[f'{measure}_변화량'] = change
                        progress_info[f'{measure}_성장률'] = growth_rate
                    
                    # 시계열 패턴 - 주기적으로 측정값 추출 (최대 5개 지점)
                    if len(valid_visits) >= 5:
                        sample_indices = np.linspace(0, len(valid_visits)-1, 5, dtype=int)
                        sample_values = [valid_visits[measure].iloc[i] for i in sample_indices]
                        progress_info[f'{measure}_샘플값'] = sample_values
                    
                    # 선형 추세선 계산
                    if len(valid_visits) >= 3:
                        x = np.arange(len(valid_visits))
                        y = valid_visits[measure].values
                        slope, intercept = np.polyfit(x, y, 1)
                        
                        progress_info[f'{measure}_추세선_기울기'] = slope
                        progress_info[f'{measure}_추세선_절편'] = intercept
                        
                        # 최근 경향 (마지막 3회 방문)
                        if len(valid_visits) >= 3:
                            recent_visits = valid_visits.iloc[-3:]
                            recent_slope, recent_intercept = np.polyfit(
                                np.arange(len(recent_visits)), 
                                recent_visits[measure].values, 
                                1
                            )
                            progress_info[f'{measure}_최근추세_기울기'] = recent_slope
        
        # 환자 진행 정보 저장
        patient_progress[patient_id] = progress_info
    
    # 딕셔너리를 데이터프레임으로 변환
    progress_df = pd.DataFrame.from_dict(list(patient_progress.values()))
    
    return progress_df

### 군집화

In [11]:
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
## 사용
def load_and_process_patient_embeddings(embedding_files):
    """리스트 구조의 임베딩 파일을 적절히 처리하는 함수"""
    patient_embeddings = {}
    
    for file_path in embedding_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # 파일이 리스트 형태인 경우
            if isinstance(data, list):
                for item in data:
                    if 'patient_id' in item and 'embedding' in item:
                        pid = item['patient_id']
                        # 기존 환자 데이터가 있으면 리스트에 추가
                        if pid in patient_embeddings:
                            if isinstance(patient_embeddings[pid], list):
                                patient_embeddings[pid].append(item)
                            else:
                                patient_embeddings[pid] = [patient_embeddings[pid], item]
                        else:
                            # 처음 등장하는 환자는 리스트로 바로 저장
                            patient_embeddings[pid] = [item]
    
    return patient_embeddings

def create_patient_similarity_matrix(patient_embeddings):
    """환자 임베딩 간의 유사도 행렬 생성"""
    patient_ids = list(patient_embeddings.keys())
    embeddings = np.array([patient_embeddings[pid]['embedding'] for pid in patient_ids])
    
    # 코사인 유사도 계산 (1에 가까울수록 유사)
    similarity_matrix = 1 - cosine_similarity(embeddings)
    
    return patient_ids, similarity_matrix

# 사용
def cluster_patients_by_embeddings(patient_embeddings, n_clusters=5, method='kmeans'):
    """임베딩 벡터를 사용한 환자 군집화 - 데이터 구조 유연성 추가"""
    # 환자 임베딩 추출 - 다양한 데이터 구조 지원
    patient_ids = []
    embeddings_list = []
    
    # 딕셔너리 키-값 구조인 경우
    if isinstance(patient_embeddings, dict):
        for pid, data in patient_embeddings.items():
            # 값이 딕셔너리인 경우
            if isinstance(data, dict) and 'embedding' in data:
                patient_ids.append(pid)
                embeddings_list.append(data['embedding'])
            # 값이 리스트인 경우 (첫 번째 요소를 임베딩으로 가정)
            elif isinstance(data, list) and len(data) > 0:
                patient_ids.append(pid)
                embeddings_list.append(data[0] if isinstance(data[0], (list, np.ndarray)) else data)
    
    # 리스트 구조인 경우
    elif isinstance(patient_embeddings, list):
        for item in patient_embeddings:
            if isinstance(item, dict) and 'patient_id' in item and 'embedding' in item:
                patient_ids.append(item['patient_id'])
                embeddings_list.append(item['embedding'])
    
    # 충분한 데이터가 없으면 빈 결과 반환
    if len(embeddings_list) < n_clusters:
        print(f"군집화를 위한 충분한 데이터가 없습니다. 환자 수: {len(embeddings_list)}")
        return {}, {}
    
    # 임베딩을 numpy 배열로 변환
    try:
        embeddings = np.array(embeddings_list)
    except ValueError as e:
        print(f"임베딩 배열 변환 오류: {e}")
        print(f"첫 번째 임베딩 타입: {type(embeddings_list[0])}")
        # 차원이 일관되지 않은 경우 차원 맞추기 시도
        max_dim = max(len(emb) for emb in embeddings_list if hasattr(emb, '__len__'))
        embeddings = np.array([
            np.pad(emb, (0, max_dim - len(emb))) if hasattr(emb, '__len__') and len(emb) < max_dim else emb
            for emb in embeddings_list
        ])
    
    # 군집화 실행
    if method == 'kmeans':
        # K-means 군집화
        kmeans = KMeans(n_clusters=min(n_clusters, len(embeddings)), random_state=42)
        clusters = kmeans.fit_predict(embeddings)
    elif method == 'dbscan':
        # DBSCAN 군집화 (밀도 기반)
        dbscan = DBSCAN(eps=0.3, min_samples=5)
        clusters = dbscan.fit_predict(embeddings)
    else:
        raise ValueError(f"지원되지 않는 군집화 방법: {method}")
    
    # 환자 ID와 군집 번호 매핑
    patient_clusters = {patient_ids[i]: int(clusters[i]) for i in range(len(patient_ids))}
    
    # 군집별 환자 수 계산
    cluster_counts = {}
    for cluster_id in set(clusters):
        if cluster_id != -1:  # DBSCAN의 노이즈 포인트 제외
            count = list(clusters).count(cluster_id)
            cluster_counts[cluster_id] = count
    
    print(f"군집화 방법: {method}, 군집 수: {len(cluster_counts)}")
    for cluster_id, count in cluster_counts.items():
        print(f"  - 군집 {cluster_id}: {count}명의 환자")
    
    return patient_clusters, cluster_counts

def visualize_patient_embeddings(patient_embeddings, patient_clusters, recovered_ids=None):
    """t-SNE를 사용하여 환자 임베딩 2D 시각화 - 리스트 구조 지원 강화"""
    # 빈 임베딩 처리
    if not patient_embeddings:
        print("임베딩 데이터가 없어 시각화를 건너뜁니다.")
        return None
    
    # 임베딩 벡터 추출을 위한 준비
    patient_ids = []
    embeddings_list = []
    
    # 각 환자별로 임베딩 추출 시도
    for pid, data in patient_embeddings.items():
        embedding_vector = None
        data_source = "알 수 없음"
        
        # 1. 리스트 구조 처리
        if isinstance(data, list):
            if len(data) > 0:
                first_item = data[0]
                # 1-1. 리스트의 첫 항목이 딕셔너리인 경우
                if isinstance(first_item, dict):
                    for key in ['embedding', 'sequence_embedding', 'vector']:
                        if key in first_item:
                            embedding_vector = first_item[key]
                            data_source = f"리스트 내 딕셔너리의 '{key}' 필드"
                            break
                # 1-2. 리스트의 첫 항목이 직접 임베딩 벡터인 경우
                elif isinstance(first_item, (list, np.ndarray)):
                    embedding_vector = first_item
                    data_source = "리스트의 첫 항목(벡터)"
        
        # 2. 딕셔너리 구조 처리
        elif isinstance(data, dict):
            for key in ['embedding', 'sequence_embedding', 'vector']:
                if key in data:
                    embedding_vector = data[key]
                    data_source = f"딕셔너리의 '{key}' 필드"
                    break
        
        # 3. 추출된 임베딩 검증 및 추가
        if embedding_vector is not None:
            # 임베딩이 배열 또는 리스트 형태인지 확인
            if hasattr(embedding_vector, '__len__'):
                try:
                    # 간단한 벡터 유효성 검사 (최소 차원 확인)
                    if len(embedding_vector) >= 10:  # 최소 10차원 이상 필요하다고 가정
                        patient_ids.append(pid)
                        embeddings_list.append(embedding_vector)
                        print(f"환자 {pid}: {data_source}에서 {len(embedding_vector)}차원 임베딩 추출 성공")
                    else:
                        print(f"환자 {pid}: 임베딩 차원이 너무 작음 ({len(embedding_vector)})")
                except Exception as e:
                    print(f"환자 {pid}: 임베딩 벡터 검증 중 오류 - {e}")
        else:
            print(f"환자 {pid}: 임베딩을 찾을 수 없음")
    
    # 추출된 임베딩이 없는 경우
    if not embeddings_list:
        print("시각화할 유효한 임베딩 데이터가 없습니다.")
        print(f"처리된 환자 수: {len(patient_embeddings)}, 추출된 임베딩 수: 0")
        return None
        
    print(f"시각화를 위해 {len(embeddings_list)}개의 임베딩이 준비되었습니다.")
    
    try:
        # 임베딩 배열 생성 - 차원 불일치 시 안전하게 처리
        max_dim = max(len(emb) for emb in embeddings_list)
        print(f"최대 임베딩 차원: {max_dim}")
        
        # 차원 맞추기
        uniform_embeddings = []
        for emb in embeddings_list:
            if len(emb) < max_dim:
                # 작은 차원의 임베딩은 0으로 패딩
                padded_emb = np.pad(emb, (0, max_dim - len(emb)), 'constant')
                uniform_embeddings.append(padded_emb)
            else:
                uniform_embeddings.append(emb)
        
        embeddings = np.array(uniform_embeddings)
        
        # 고차원 임베딩을 2차원으로 축소
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1), n_iter=1000)
        embeddings_2d = tsne.fit_transform(embeddings)
        
        # 시각화
        plt.figure(figsize=(12, 10))
        
        # 각 군집별로 색상 지정 - 군집이 비어있으면 기본 처리
        if not patient_clusters:
            plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.7)
            plt.title('t-SNE로 시각화한 환자 임베딩 (군집 없음)')
        else:
            unique_clusters = sorted(set(patient_clusters.values()))
            colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_clusters) if unique_clusters else 1))
            
            # 군집별로 플롯
            for i, cluster_id in enumerate(unique_clusters):
                if cluster_id == -1:  # DBSCAN 노이즈 포인트
                    continue
                    
                # 현재 군집에 속한 환자 인덱스
                indices = [j for j, pid in enumerate(patient_ids) if pid in patient_clusters and patient_clusters[pid] == cluster_id]
                
                if indices:  # 인덱스가 있는 경우에만 플롯
                    # 군집 데이터 플롯
                    plt.scatter(
                        embeddings_2d[indices, 0], 
                        embeddings_2d[indices, 1],
                        c=[colors[i]],
                        label=f'군집 {cluster_id} ({len(indices)}명)',
                        alpha=0.7
                    )
        
        # 완치 환자 강조 표시 (있는 경우)
        if recovered_ids:
            recovered_indices = [i for i, pid in enumerate(patient_ids) if pid in recovered_ids]
            if recovered_indices:  # 완치 환자가 있는 경우에만 플롯
                plt.scatter(
                    embeddings_2d[recovered_indices, 0],
                    embeddings_2d[recovered_indices, 1],
                    c='black',
                    marker='*',
                    s=150,
                    label=f'완치 환자 ({len(recovered_indices)}명)',
                    alpha=0.8
                )
        
        plt.title('t-SNE로 시각화한 환자 임베딩')
        plt.xlabel('t-SNE 차원 1')
        plt.ylabel('t-SNE 차원 2')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        
        return embeddings_2d
    
    except Exception as e:
        print(f"임베딩 시각화 중 오류 발생: {e}")
        # 스택 트레이스 출력 (선택적)
        import traceback
        traceback.print_exc()
        return None

### 예후예측 모델

In [12]:
def identify_recovered_patients_improved(patient_progress, threshold=70):
    """
    개선된 복합 완치 점수를 기반으로 회복된 환자 식별
    
    Parameters:
    -----------
    patient_progress : pandas.DataFrame
        환자별 첫 방문과 마지막 방문 데이터가 포함된 데이터프레임
    threshold : float, optional (default=70)
        완치로 간주할 복합 점수 임계값
    
    Returns:
    --------
    list
        회복된 환자 ID 목록
    dict
        각 회복 등급별 환자 수 통계
    dict
        환자별 세부 회복 정보
    """
    # 결과 저장할 컨테이너
    recovered_patients = []
    patient_details = {}
    recovery_grades = {
        "완전 회복 (Complete Recovery)": 0,
        "상당한 회복 (Substantial Recovery)": 0,
        "중간 회복 (Moderate Recovery)": 0,
        "경미한 회복 (Mild Recovery)": 0,
        "최소 회복 (Minimal Recovery)": 0,
        "회복 미미 (Little to No Recovery)": 0
    }
    
    # 각 환자에 대해 복합 점수 계산
    for _, row in patient_progress.iterrows():
        patient_id = row['환자번호']
        
        # 복합 완치 점수 계산
        composite_score, details = calculate_composite_recovery_score(row)
        
        # 상세 정보 저장
        patient_details[patient_id] = {
            'composite_score': composite_score,
            'recovery_grade': details['recovery_grade'],
            'score_breakdown': details['scores']
        }
        
        # 회복 등급 통계 업데이트
        recovery_grades[details['recovery_grade']] += 1
        
        # 임계값 이상인 환자는 회복된 것으로 간주
        if composite_score >= threshold:
            recovered_patients.append(patient_id)
    
    # 결과 출력
    print(f"복합 완치 점수 {threshold} 이상인 환자: {len(recovered_patients)}명")
    print(f"회복 등급 분포:")
    for grade, count in recovery_grades.items():
        print(f"  - {grade}: {count}명")
    
    return recovered_patients, recovery_grades, patient_details

In [13]:
def predict_patient_recovery(patient_data, patient_embeddings, recovery_prototype, trajectory_database, embedding_analysis=None):
    """두 예측 함수의 장점을 통합한 향상된 완치 예측 함수 - 리스트 구조 지원 추가"""
    patient_id = patient_data['환자번호']
    
    # 기본 결과 템플릿 (오류 처리용)
    default_result = {
        'patient_id': patient_id,
        'prediction_possible': False,
        'reason': "알 수 없는 오류",
        'similarity_to_recovery_prototype': 0.0,
        'predicted_trajectory': "예측 불가",
        'estimated_days_to_recovery': 0,
        'predicted_composite_score': 0.0,
        'predicted_recovery_grade': "예측 불가",
        'confidence_score': 0.0
    }
    
    try:
        # 환자 임베딩 확인
        if patient_id not in patient_embeddings:
            default_result.update({'reason': "환자 임베딩 데이터 없음"})
            return default_result
        
        # 회복 프로토타입 확인
        if recovery_prototype is None:
            default_result.update({'reason': "완치 환자 프로토타입 없음"})
            return default_result
        
        # 이미 분석된 환자인지 확인
        if embedding_analysis and patient_id in embedding_analysis:
            current_details = embedding_analysis[patient_id]
            
            # 이미 높은 완치 점수를 가진 경우
            if current_details['composite_score'] >= 70:
                return {
                    'patient_id': patient_id,
                    'already_recovered': True,
                    'recovery_grade': current_details['recovery_grade'],
                    'composite_score': current_details['composite_score'],
                    'similarity_to_recovery_prototype': current_details.get('similarity_to_recovery_prototype', 0.0),
                    'predicted_trajectory': "이미 회복됨",
                    'estimated_days_to_recovery': 0,
                    'predicted_composite_score': current_details['composite_score'],
                    'confidence_score': 100.0
                }
        
        # 임베딩 추출 - 리스트 구조 지원
        patient_embedding = None
        data = patient_embeddings[patient_id]
        
        # 리스트 구조 처리 (검증 결과와 일치)
        if isinstance(data, list) and len(data) > 0:
            # 리스트의 첫 항목이 딕셔너리인 경우
            if isinstance(data[0], dict) and 'embedding' in data[0]:
                patient_embedding = data[0]['embedding']
            # 리스트의 첫 항목이 임베딩 벡터인 경우
            elif isinstance(data[0], (list, np.ndarray)):
                patient_embedding = data[0]
        # 딕셔너리 구조 처리 (기존 코드)
        elif isinstance(data, dict):
            if 'embedding' in data:
                patient_embedding = data['embedding']
            elif 'sequence_embedding' in data:
                patient_embedding = data['sequence_embedding']
        
        # 임베딩이 없는 경우
        if patient_embedding is None:
            default_result.update({'reason': "환자 임베딩을 추출할 수 없음"})
            return default_result
        
        # 프로토타입과의 유사도 계산
        try:
            similarity_to_prototype = 1 - cosine(patient_embedding, recovery_prototype)
        except Exception as e:
            print(f"유사도 계산 오류: {e}")
            similarity_to_prototype = 0.0
        
        # 1. 궤적 기반 유사 환자 찾기
        similar_patients_by_trajectory = []
        similar_patients_by_embedding = []
        
        # 궤적 정보 기반 유사 환자 찾기
        for pid, trajectories in trajectory_database.items():
            if pid == patient_id:
                continue
            
            # 환자의 궤적이 유사한지 확인
            similarity_score = 0
            trajectory_models = {}
            
            # 리스트 구조에서 trajectory_models 추출
            if isinstance(patient_embeddings[patient_id], list) and len(patient_embeddings[patient_id]) > 0:
                if isinstance(patient_embeddings[patient_id][0], dict):
                    trajectory_models = patient_embeddings[patient_id][0].get('trajectory_models', {})
            # 딕셔너리 구조에서 trajectory_models 추출
            elif isinstance(patient_embeddings[patient_id], dict):
                trajectory_models = patient_embeddings[patient_id].get('trajectory_models', {})
            
            # 유사도 점수 계산
            for measure in ['CC_vas', 'CMO_before', 'MMO_before']:
                if (measure in trajectories and measure in trajectory_models):
                    similarity_score += 0.3
            
            if similarity_score > 0:
                similar_patients_by_trajectory.append((pid, similarity_score))
        
        # 2. 임베딩 기반 완치 환자 찾기
        if embedding_analysis:
            for pid, details in embedding_analysis.items():
                if details.get('composite_score', 0) >= 70:  # 완치 환자만
                    try:
                        # 다른 환자의 임베딩 추출 - 리스트 구조 지원
                        other_embedding = None
                        other_data = patient_embeddings[pid]
                        
                        # 리스트 구조 처리
                        if isinstance(other_data, list) and len(other_data) > 0:
                            if isinstance(other_data[0], dict) and 'embedding' in other_data[0]:
                                other_embedding = other_data[0]['embedding']
                            elif isinstance(other_data[0], (list, np.ndarray)):
                                other_embedding = other_data[0]
                        # 딕셔너리 구조 처리
                        elif isinstance(other_data, dict):
                            if 'embedding' in other_data:
                                other_embedding = other_data['embedding']
                            elif 'sequence_embedding' in other_data:
                                other_embedding = other_data['sequence_embedding']
                        
                        # 임베딩이 있는 경우에만 유사도 계산
                        if other_embedding is not None:
                            similarity = 1 - cosine(patient_embedding, other_embedding)
                            similar_patients_by_embedding.append((pid, similarity))
                    except Exception as e:
                        print(f"환자 {pid}와의 유사도 계산 오류: {e}")
                        continue
        
        # 두 유사 환자 목록 통합 (중복 제거 및 점수 합산)
        combined_similar_patients = {}
        
        # 궤적 기반 유사 환자 추가
        for pid, score in similar_patients_by_trajectory:
            combined_similar_patients[pid] = score
        
        # 임베딩 기반 유사 환자 추가 (점수 합산)
        for pid, score in similar_patients_by_embedding:
            if pid in combined_similar_patients:
                combined_similar_patients[pid] += score
            else:
                combined_similar_patients[pid] = score
        
        # 상위 유사 환자 선택
        similar_patients = [(pid, score) for pid, score in combined_similar_patients.items()]
        similar_patients.sort(key=lambda x: x[1], reverse=True)
        top_similar = similar_patients[:5] if similar_patients else []
        
        # 유사 환자가 없는 경우
        if not top_similar:
            default_result.update({
                'reason': "유사한 환자 데이터 없음",
                'similarity_to_recovery_prototype': similarity_to_prototype
            })
            return default_result
        
        # 유사 환자들의 회복 궤적 분석
        trajectory_types = []
        avg_recovery_days = 0
        avg_daily_changes = {
            'vas_daily_change': 0,
            'cmo_daily_change': 0,
            'mmo_daily_change': 0
        }
        
        similar_count = 0
        for pid, similarity in top_similar:
            # 궤적 데이터베이스에서 정보 추출
            if pid in trajectory_database:
                for measure, model in trajectory_database[pid].items():
                    if model and 'trajectory_type' in model:
                        trajectory_types.append(model['trajectory_type'])
                        
                    # 예측 일수 추출
                    if model and 'predicted_days_to_target' in model and model['predicted_days_to_target']:
                        avg_recovery_days += model['predicted_days_to_target']
            
            # 시계열 특성 분석 - 리스트 구조 지원
            time_features = {}
            
            # 리스트 구조에서 time_features 추출
            if pid in patient_embeddings:
                if isinstance(patient_embeddings[pid], list) and len(patient_embeddings[pid]) > 0:
                    if isinstance(patient_embeddings[pid][0], dict):
                        time_features = patient_embeddings[pid][0].get('time_features', {})
                # 딕셔너리 구조에서 time_features 추출
                elif isinstance(patient_embeddings[pid], dict):
                    time_features = patient_embeddings[pid].get('time_features', {})
            
            # 일일 변화율 계산 (시계열 특성이 있는 경우만)
            if time_features:
                for feature, value in time_features.items():
                    if feature.endswith('_daily_change'):
                        change_type = feature
                        if change_type in avg_daily_changes:
                            avg_daily_changes[change_type] += value
                
                similar_count += 1
        
        # 평균 계산 (0으로 나누기 방지)
        if similar_count > 0:
            avg_recovery_days /= similar_count
            for change_type in avg_daily_changes.keys():
                avg_daily_changes[change_type] /= similar_count
        
        # 가장 흔한 회복 궤적 유형
        if trajectory_types:
            try:
                from collections import Counter
                most_common_trajectory = Counter(trajectory_types).most_common(1)[0][0]
            except:
                most_common_trajectory = "알 수 없음"
        else:
            most_common_trajectory = "데이터 부족"
        
        # 유사도 기반 회복 기간 조정
        recovery_time_factor = 1 + (1 - similarity_to_prototype) * 0.7
        estimated_days = max(1, avg_recovery_days * recovery_time_factor)
        
        # 현재 임상 지표
        current_vas = patient_data.get('CC_vas', None)
        current_cmo = patient_data.get('CMO_before', None)
        current_mmo = patient_data.get('MMO_before', None)
        
        # 예상 최종 상태 계산
        predicted_values = {}
        if pd.notna(current_vas):
            predicted_values['final_vas'] = max(0, current_vas + avg_daily_changes['vas_daily_change'] * estimated_days)
        if pd.notna(current_cmo):
            predicted_values['final_cmo'] = current_cmo + avg_daily_changes['cmo_daily_change'] * estimated_days
        if pd.notna(current_mmo):
            predicted_values['final_mmo'] = current_mmo + avg_daily_changes['mmo_daily_change'] * estimated_days
        
        # 예상 완치 점수 계산
        predicted_row = patient_data.copy()
        if 'final_vas' in predicted_values:
            predicted_row['CC_vas'] = predicted_values['final_vas']
        if 'final_cmo' in predicted_values:
            predicted_row['CMO_before'] = predicted_values['final_cmo']
        if 'final_mmo' in predicted_values:
            predicted_row['MMO_before'] = predicted_values['final_mmo']
        
        try:
            predicted_score, details = calculate_composite_recovery_score(predicted_row)
            predicted_grade = details['recovery_grade']
        except Exception as e:
            print(f"완치 점수 계산 오류: {e}")
            predicted_score = 0
            predicted_grade = "예측 불가"
        
        # 시계열 기반 예측 개선 - 리스트 구조 지원
        confidence_boost = 0
        try:
            # 환자의 시계열 특성 추출 - 리스트 구조 지원
            time_features = {}
            
            # 리스트 구조에서 시계열 특성 추출
            if isinstance(patient_embeddings[patient_id], list) and len(patient_embeddings[patient_id]) > 0:
                if isinstance(patient_embeddings[patient_id][0], dict):
                    time_features = patient_embeddings[patient_id][0].get('time_features', {})
            # 딕셔너리 구조에서 시계열 특성 추출
            elif isinstance(patient_embeddings[patient_id], dict):
                time_features = patient_embeddings[patient_id].get('time_features', {})
            
            # 시계열 특성이 있는 경우만 처리
            if time_features:
                # 방문 횟수에 따른 예측 신뢰도 조정
                visit_count = time_features.get('방문횟수', 1)
                if visit_count > 1:
                    confidence_boost = min(20, (visit_count - 1) * 5)  # 최대 20%까지 신뢰도 상승
                
                # 추세 기울기 활용
                for measure in ['CC_vas_추세_기울기', 'CMO_before_추세_기울기', 'MMO_before_추세_기울기']:
                    if measure in time_features:
                        trend_slope = time_features[measure]
                        
                        # 개선 여부 판단
                        is_improving = (measure == 'CC_vas_추세_기울기' and trend_slope < 0) or \
                                      (measure != 'CC_vas_추세_기울기' and trend_slope > 0)
                        
                        if is_improving:
                            estimated_days *= 0.9  # 개선 추세면 회복 기간 단축
                        else:
                            estimated_days *= 1.1  # 악화 추세면 회복 기간 연장
                
                # 궤적 유형 업데이트
                vas_slope = time_features.get('CC_vas_추세_기울기', 0)
                cmo_slope = time_features.get('CMO_before_추세_기울기', 0)
                
                # 기울기 기반 패턴 분류
                if vas_slope < -0.2 and cmo_slope > 0.2:
                    trajectory_pattern = "빠른 전반적 개선형"
                elif vas_slope < -0.1:
                    trajectory_pattern = "통증 우선 개선형"
                elif cmo_slope > 0.1:
                    trajectory_pattern = "기능 우선 개선형"
                elif -0.1 <= vas_slope < 0 or 0 < cmo_slope <= 0.1:
                    trajectory_pattern = "완만한 개선형"
                elif vas_slope > 0 or cmo_slope < 0:
                    trajectory_pattern = "악화 가능성 있음"
                else:
                    trajectory_pattern = most_common_trajectory
                
                # 유효한 패턴인 경우만 업데이트
                if trajectory_pattern != "알 수 없음" and trajectory_pattern != "데이터 부족":
                    most_common_trajectory = trajectory_pattern
        except Exception as e:
            print(f"시계열 기반 예측 개선 중 오류: {e}")
        
        # 결과 반환 - 시계열 특성 추출 추가
        time_features_result = {}
        non_linear_patterns = {}
        
        # 리스트 구조에서 특성 추출
        if isinstance(patient_embeddings[patient_id], list) and len(patient_embeddings[patient_id]) > 0:
            if isinstance(patient_embeddings[patient_id][0], dict):
                time_features_result = patient_embeddings[patient_id][0].get('time_features', {})
                non_linear_patterns = patient_embeddings[patient_id][0].get('non_linear_patterns', {})
        # 딕셔너리 구조에서 특성 추출
        elif isinstance(patient_embeddings[patient_id], dict):
            time_features_result = patient_embeddings[patient_id].get('time_features', {})
            non_linear_patterns = patient_embeddings[patient_id].get('non_linear_patterns', {})
        
        # 최종 결과 반환
        return {
            'patient_id': patient_id,
            'prediction_possible': True,
            'similarity_to_recovery_prototype': similarity_to_prototype,
            'similar_patients': [pid for pid, _ in top_similar],
            'similarity_scores': [similarity for _, similarity in top_similar],
            'predicted_trajectory': most_common_trajectory,
            'estimated_days_to_recovery': round(estimated_days),
            'avg_daily_changes': avg_daily_changes,
            'predicted_final_values': predicted_values,
            'predicted_composite_score': predicted_score,
            'predicted_recovery_grade': predicted_grade,
            'confidence_score': min(100, (similarity_to_prototype * 100) + confidence_boost),
            'time_features': time_features_result,
            'non_linear_patterns': non_linear_patterns
        }
    
    except Exception as e:
        # 모든 예외 상황 처리
        default_result.update({
            'reason': f"예측 오류: {str(e)}"
        })
        return default_result

### 임베딩 클러스터와 완치 환자 관계 분석

In [14]:
def analyze_clusters_and_recovery(patient_clusters, embedding_analysis, recovery_threshold=70):
    """
    군집별 완치 환자 분포 및 특성 분석
    
    Parameters:
    -----------
    patient_clusters : dict
        환자ID를 키로, 군집 번호를 값으로 하는 딕셔너리
    embedding_analysis : dict
        환자별 임베딩 분석 결과
    recovery_threshold : float, optional (default=70)
        완치로 간주할 복합 점수 임계값
    
    Returns:
    --------
    dict
        군집별 완치 환자 분석 결과
    """
    # 군집별 환자 및 완치 환자 수 계산
    cluster_analysis = {}
    
    # 각 군집별로 분석
    for cluster_id in set(patient_clusters.values()):
        # 현재 군집에 속한 환자
        cluster_patients = [pid for pid, cid in patient_clusters.items() if cid == cluster_id]
        
        # 군집 내 환자 중 완치 점수 있는 환자
        analyzed_patients = [pid for pid in cluster_patients if pid in embedding_analysis]
        
        if not analyzed_patients:
            continue
        
        # 완치된 환자 수
        recovered_patients = [
            pid for pid in analyzed_patients 
            if embedding_analysis[pid]['composite_score'] >= recovery_threshold
        ]
        
        # 군집 내 완치율
        recovery_rate = len(recovered_patients) / len(analyzed_patients) if analyzed_patients else 0
        
        # 군집 내 평균 완치 점수
        avg_recovery_score = sum(
            embedding_analysis[pid]['composite_score'] for pid in analyzed_patients
        ) / len(analyzed_patients) if analyzed_patients else 0
        
        # 완치 등급 분포
        grade_distribution = {}
        for pid in analyzed_patients:
            grade = embedding_analysis[pid]['recovery_grade']
            grade_distribution[grade] = grade_distribution.get(grade, 0) + 1
        
        # 분석 결과 저장
        cluster_analysis[cluster_id] = {
            'total_patients': len(cluster_patients),
            'analyzed_patients': len(analyzed_patients),
            'recovered_patients': len(recovered_patients),
            'recovery_rate': recovery_rate,
            'avg_recovery_score': avg_recovery_score,
            'grade_distribution': grade_distribution
        }
    
    # 결과 출력
    print(f"군집별 완치 분석 결과 (완치 기준 점수: {recovery_threshold})")
    for cluster_id, analysis in cluster_analysis.items():
        print(f"\n군집 {cluster_id} 분석:")
        print(f"  - 총 환자 수: {analysis['total_patients']}명")
        print(f"  - 분석된 환자 수: {analysis['analyzed_patients']}명")
        print(f"  - 완치 환자 수: {analysis['recovered_patients']}명")
        print(f"  - 완치율: {analysis['recovery_rate']*100:.1f}%")
        print(f"  - 평균 완치 점수: {analysis['avg_recovery_score']:.1f}")
        print("  - 등급 분포:")
        for grade, count in analysis['grade_distribution'].items():
            percent = count / analysis['analyzed_patients'] * 100
            print(f"    · {grade}: {count}명 ({percent:.1f}%)")
    
    return cluster_analysis

### 각 기능 통합

In [15]:
def integrated_recovery_prediction_system(df, embedding_files, recovery_threshold=70, n_clusters=5):
    """임베딩과 환자 시계열 데이터를 통합한 완치 예측 시스템"""
    
    # 1. 환자별 전체 방문 데이터 추출 (시계열 특성 포함)
    print("환자별 방문 데이터 추출 및 시계열 특성 분석 중...")
    patient_progress = extract_patient_progress(df)
    
    # 2. 임베딩 데이터 로드 또는 생성
    print("\n임베딩 데이터 로드 중...")
    
    # 임베딩 파일 존재 여부 확인
    embedding_files_exist = all(os.path.exists(f) for f in embedding_files)
    
    if embedding_files_exist:
        # 수정된 로드 함수 사용
        patient_embeddings = load_and_process_patient_embeddings(embedding_files)
        
        # 필요한 형식으로 데이터 구조 변환 확인
        if not isinstance(patient_embeddings, dict):
            print("임베딩 데이터 구조 변환 중...")
            temp_embeddings = {}
            for item in patient_embeddings:
                if isinstance(item, dict) and 'patient_id' in item:
                    temp_embeddings[item['patient_id']] = item
            patient_embeddings = temp_embeddings
    else:
        print("임베딩 파일이 존재하지 않습니다. 새로 생성합니다...")
        patient_embeddings = create_hybrid_contextual_embedding(df, api_key)
        
        # 생성된 임베딩 저장
        save_dir = '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings'
        os.makedirs(save_dir, exist_ok=True)
        
        save_embeddings_to_json(
            list(patient_embeddings.values()), 
            os.path.join(save_dir, 'patient_time_series_embeddings.json')
        )
    
    # 3. 환자 궤적 분석 및 데이터베이스 구축
    print("\n환자 회복 궤적 분석 중...")
    trajectory_database = {}
    
    for patient_id, patient_visits in df.groupby('환자번호'):
        if len(patient_visits) >= 2:
            trajectory_models = {}
            for measure in ['CC_vas', 'CMO_before', 'MMO_before']:
                if measure in patient_visits.columns:
                    trajectory_models[measure] = model_recovery_trajectory(patient_visits, measure)
            
            trajectory_database[patient_id] = trajectory_models
    
    print(f"총 {len(trajectory_database)}명의 환자 궤적 분석 완료")
    
    # 4. 군집화 및 완치 환자 식별
    print("\n환자 임베딩 군집화 및 완치 환자 식별 중...")
    
    # 임베딩 벡터 추출
    # 임베딩 벡터 추출 - 리스트 구조 지원 추가
    embedding_vectors = {}

    for pid, data in patient_embeddings.items():
        # 환자 데이터가 리스트인 경우 (검증 결과와 일치)
        if isinstance(data, list) and len(data) > 0:
            # 리스트의 첫 항목이 딕셔너리인 경우
            if isinstance(data[0], dict) and 'embedding' in data[0]:
                embedding_vectors[pid] = data[0]['embedding']
                print(f"환자 {pid}: 리스트 내 딕셔너리의 'embedding' 필드 사용")
            # 리스트의 첫 항목이 임베딩 벡터인 경우
            elif isinstance(data[0], (list, np.ndarray)):
                embedding_vectors[pid] = data[0]
                print(f"환자 {pid}: 리스트의 첫 항목을 임베딩으로 사용")
        # 기존 코드: 환자 데이터가 딕셔너리인 경우
        elif isinstance(data, dict):
            if 'embedding' in data:
                embedding_vectors[pid] = data['embedding']
                print(f"환자 {pid}: 딕셔너리의 'embedding' 필드 사용")
            elif 'sequence_embedding' in data:
                embedding_vectors[pid] = data['sequence_embedding']
                print(f"환자 {pid}: 딕셔너리의 'sequence_embedding' 필드 사용")

    print(f"추출된 임베딩 벡터 수: {len(embedding_vectors)}")
    
    # 환자 군집화
    patient_clusters, cluster_counts = cluster_patients_by_embeddings(
        embedding_vectors, n_clusters, method='kmeans'
    )
    
    # 완치 환자 식별
    recovered_ids = []
    recovery_analysis = {}
    
    for idx, row in patient_progress.iterrows():
        patient_id = row['환자번호']
        
        # 복합 완치 점수 계산
        composite_score, details = calculate_composite_recovery_score(row)
        
        # 분석 결과 저장
        recovery_analysis[patient_id] = {
            'composite_score': composite_score,
            'recovery_grade': details['recovery_grade'],
            'score_breakdown': details['scores']
        }
        
        # 임계값 이상인 환자는 회복된 것으로 간주
        if composite_score >= recovery_threshold:
            recovered_ids.append(patient_id)
    
    # 5. 완치 환자 임베딩 프로토타입 계산
    recovery_prototype = None
    if recovered_ids:
        prototype_vectors = [embedding_vectors[pid] for pid in recovered_ids if pid in embedding_vectors]
        if prototype_vectors:
            recovery_prototype = np.mean(prototype_vectors, axis=0)
    
    # 6. 군집별 완치 패턴 분석
    print("\n군집별 완치 패턴 분석 중...")
    cluster_analysis = analyze_clusters_and_recovery(
        patient_clusters, recovery_analysis, recovery_threshold
    )
    
    # 7. 임베딩 시각화
    print("\n임베딩 시각화 중...")
    embeddings_2d = visualize_patient_embeddings(
        embedding_vectors, patient_clusters, recovered_ids
    )
    
    # 8. 완치 예측 시스템 준비
    print("\n완치 예측 시스템 준비 완료!")
    
    # 통합 결과 반환
    return {
        'patient_embeddings': patient_embeddings,
        'trajectory_database': trajectory_database,
        'recovered_ids': recovered_ids,
        'recovery_prototype': recovery_prototype,
        'recovery_analysis': recovery_analysis,
        'patient_clusters': patient_clusters,
        'cluster_analysis': cluster_analysis,
        'embeddings_2d': embeddings_2d,
        
        # 예측 함수 - 구버전에서 통합된 새 함수로 변경
        'predict_recovery': lambda patient_data: predict_patient_recovery(
            patient_data, 
            patient_embeddings, 
            recovery_prototype, 
            trajectory_database,
            recovery_analysis  # 신버전에서 사용하는 embedding_analysis 추가
        )
    }

## 실행

- 임베딩 생성

In [ ]:
sample_patients = df.sample(20)['환자번호'].unique().tolist()
df_sample = df[df['환자번호'].isin(sample_patients)]
batch_size = 20

embedding_dir = '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings'
embedding_files = [os.path.join(embedding_dir, f) for f in os.listdir(embedding_dir) 
                  if f.endswith('.json')]


do_create_embeddings(df_sample, batch_size)

- 예측 수행

In [16]:
prediction_system = integrated_recovery_prediction_system(
    df_sample,
    embedding_files=embedding_files,
    recovery_threshold=70,
    n_clusters=5
)


환자별 방문 데이터 추출 및 시계열 특성 분석 중...

임베딩 데이터 로드 중...

환자 회복 궤적 분석 중...
총 20명의 환자 궤적 분석 완료

환자 임베딩 군집화 및 완치 환자 식별 중...
환자 2204-263: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2205-86: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2210-22: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2212-169: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2304-05: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2206-232: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2207-118: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2304-90: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2207-199: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2208-32: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2209-228: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2306-90: 리스트 내 딕셔너리의 'embedding' 필드 사용
환자 2308-293: 리스트 내 딕셔너리의 'embedding' 필드 사용
추출된 임베딩 벡터 수: 13
군집화 방법: kmeans, 군집 수: 5
  - 군집 0: 1명의 환자
  - 군집 1: 4명의 환자
  - 군집 2: 4명의 환자
  - 군집 3: 1명의 환자
  - 군집 4: 3명의 환자

군집별 완치 패턴 분석 중...
군집별 완치 분석 결과 (완치 기준 점수: 70)

군집 0 분석:
  - 총 환자 수: 1명
  - 분석된 환자 수: 1명
  - 완치 환자 수: 0명
  - 완치율: 0.0%
  - 평균 완치 점수: 21.5
  - 등급 분포:
    · 최소 회복 (Minimal Recovery): 1명 (100.0%)

군집 1 분석:
  - 총

In [17]:
tst = df_sample.환자번호.sample(1).values[0]
# 특정 환자의 완치 예측 수행
sample_patient = df[df['환자번호'] == tst].iloc[0]
prediction_result = prediction_system['predict_recovery'](sample_patient)

print("\n환자 분석 결과:")
print(f"환자 ID: {prediction_result.get('patient_id', '정보 없음')}")

# 예측 가능 여부 확인
if not prediction_result.get('prediction_possible', True):
    print(f"예측 불가 사유: {prediction_result.get('reason', '알 수 없음')}")
else:
    # 안전하게 값 출력 (키가 없을 경우 기본값 사용)
    print(f"완치 환자와의 유사도: {prediction_result.get('similarity_to_recovery_prototype', 0.0):.2f}")
    print(f"예상 회복 궤적: {prediction_result.get('predicted_trajectory', '예측 불가')}")
    print(f"예상 회복 소요 일수: {prediction_result.get('estimated_days_to_recovery', 0)} 일")
    print(f"예상 완치 점수: {prediction_result.get('predicted_composite_score', 0.0):.1f}")
    print(f"예상 완치 등급: {prediction_result.get('predicted_recovery_grade', '예측 불가')}")
    print(f"예측 신뢰도: {prediction_result.get('confidence_score', 0.0):.1f}%")
    
    # 이미 회복된 상태인 경우
    if prediction_result.get('already_recovered', False):
        print("참고: 이 환자는 이미 회복 상태에 도달했습니다.")

환자 2211-238와의 유사도 계산 오류: '2211-238'

환자 분석 결과:
환자 ID: 2207-199
완치 환자와의 유사도: 0.91
예상 회복 궤적: 복합 개선형 (Complex Improvement)
예상 회복 소요 일수: 63 일
예상 완치 점수: 12.5
예상 완치 등급: 회복 미미 (Little to No Recovery)
예측 신뢰도: 91.0%
